# CheMLFlow Tutorial 00: Dataset EDA

This notebook shows the simplest useful CheMLFlow workflow:

- load a local CSV
- run the generic EDA node
- inspect the generated plots

For this example, we use the bundled tutorial PGP dataset.

## 1. Choose the repo to clone

The default points at your `origin` fork and the current tutorial branch. Override `CHEMLFLOW_TUTORIAL_REPO` or `CHEMLFLOW_TUTORIAL_BRANCH` if you want a different source.

In [ ]:
import os

REPO_URL = os.environ.get("CHEMLFLOW_TUTORIAL_REPO", "https://github.com/bsmith24/CheMLFlow.git")
REPO_BRANCH = os.environ.get("CHEMLFLOW_TUTORIAL_BRANCH", "phase2_doe_support")
print(REPO_URL)
print(REPO_BRANCH)

## 2. Clone the repo and move into it

In [ ]:
%cd /content
!rm -rf CheMLFlow
!git clone --branch "$REPO_BRANCH" --single-branch "$REPO_URL"
%cd /content/CheMLFlow
!git branch --show-current
!git log -1 --oneline

## 3. Install the minimum dependencies for this tutorial

This tutorial only needs the packages required to import `main.py`, load the CSV, generate the EDA plots, and build a few RDKit-based chemistry views.

In [ ]:
!python -m pip install --upgrade pip
!python -m pip install "PyYAML>=6.0.1,<7" "pandas>=2.2.3,<2.3" "numpy>=1.26,<2.1" "scipy>=1.15.2,<1.16" "scikit-learn>=1.6.1,<1.7" "matplotlib>=3.10.1,<3.11" "seaborn>=0.13.2,<0.14" "joblib>=1.4,<2" "xgboost>=3.0.0,<3.1" "pydantic>=2.9,<3" "rdkit==2026.3.1" "plotly>=5.24,<6"


## 4. Write the config

Build the config as a normal Python object, then save it as YAML.

In [ ]:
from pathlib import Path
import yaml

CONFIG_PATH = Path("tutorials/00_dataset_eda/configs/pgp_raw_eda_colab.yaml")
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)

config = {
    "global": {
        "pipeline_type": "pgp",
        "task_type": "classification",
        "base_dir": "tutorials/00_dataset_eda/artifacts/data",
        "run_dir": "tutorials/00_dataset_eda/artifacts/run",
        "target_column": "Activity",
        "random_state": 42,
        "thresholds": {"active": 1000, "inactive": 10000},
    },
    "pipeline": {
        "nodes": ["get_data", "analyze.eda"],
    },
    "get_data": {
        "data_source": "local_csv",
        "source": {"path": "tutorials/data/pgp_broccatelli.csv"},
    },
    "analyze": {
        "eda": {
            "include": {
                "overview": True,
                "missingness": True,
                "numeric_histograms": True,
                "correlation_heatmap": True,
                "target_distribution": True,
                "class_balance": True,
                "descriptor_scatter": False,
                "descriptor_boxplots": False,
            }
        }
    },
}

config_yaml = yaml.safe_dump(config, sort_keys=False)
CONFIG_PATH.write_text(config_yaml, encoding="utf-8")
print(config_yaml)

## 5. Launch CheMLFlow

In [ ]:
import os
import subprocess
import sys

os.environ["CHEMLFLOW_CONFIG"] = str(CONFIG_PATH)
os.environ["MPLBACKEND"] = "Agg"
os.environ["MPLCONFIGDIR"] = "/content/.mplconfig"

result = subprocess.run([sys.executable, "main.py"], text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
result.check_returncode()

## 6. Inspect the manifest and generated files

In [ ]:
from pathlib import Path
import json

eda_dir = Path("tutorials/00_dataset_eda/artifacts/run/eda")
manifest = json.loads((eda_dir / "eda_manifest.json").read_text())
print(json.dumps(manifest, indent=2))
print("\nGenerated files:")
for path in sorted(eda_dir.iterdir()):
    print(path.name)

## 7. Interactive molecular landscape


In [ ]:
import base64
from io import BytesIO

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, Descriptors, Draw, Lipinski, QED
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

RDLogger.DisableLog("rdApp.*")

raw_df = pd.read_csv("tutorials/data/pgp_broccatelli.csv")
smiles_column = next((col for col in ["SMILES", "smiles", "Smiles", "canonical_smiles"] if col in raw_df.columns), None)
target_column = "Activity" if "Activity" in raw_df.columns else None
if smiles_column is None:
    raise ValueError("No SMILES column found in tutorial dataset.")

gallery_df = raw_df.copy()
gallery_df["mol"] = gallery_df[smiles_column].astype(str).apply(Chem.MolFromSmiles)
gallery_df = gallery_df[gallery_df["mol"].notna()].copy()
gallery_df["canonical_smiles"] = gallery_df["mol"].apply(Chem.MolToSmiles)
gallery_df = gallery_df.drop_duplicates(subset=["canonical_smiles"]).reset_index(drop=True)

gallery_df["molecular_weight"] = gallery_df["mol"].apply(Descriptors.MolWt)
gallery_df["logp"] = gallery_df["mol"].apply(Descriptors.MolLogP)
gallery_df["hbd"] = gallery_df["mol"].apply(Lipinski.NumHDonors)
gallery_df["hba"] = gallery_df["mol"].apply(Lipinski.NumHAcceptors)
gallery_df["qed"] = gallery_df["mol"].apply(QED.qed)

def _mol_png_data_uri(mol, size=(240, 180)):
    image = Draw.MolToImage(mol, size=size)
    buffer = BytesIO()
    image.save(buffer, format="PNG")
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    return f"data:image/png;base64,{encoded}"

def _fingerprint_array(mol, n_bits=2048):
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

fingerprint_matrix = np.vstack(gallery_df["mol"].apply(_fingerprint_array).to_list())
coords = PCA(n_components=2, random_state=42).fit_transform(fingerprint_matrix)
gallery_df["component_1"] = coords[:, 0]
gallery_df["component_2"] = coords[:, 1]
cluster_count = min(8, max(4, len(gallery_df) // 180))
gallery_df["cluster_label"] = (
    KMeans(n_clusters=cluster_count, random_state=42, n_init="auto").fit_predict(fingerprint_matrix) + 1
)

gallery_df["image_uri"] = gallery_df["mol"].apply(_mol_png_data_uri)
gallery_df["point_id"] = [f"Molecule {idx}" for idx in range(1, len(gallery_df) + 1)]
gallery_df["target_display"] = gallery_df[target_column].astype(str) if target_column else "unlabeled"
color_values = gallery_df[target_column].astype(float) if target_column else gallery_df["cluster_label"].astype(float)
color_title = target_column or "cluster_label"

customdata = gallery_df[[
    "point_id",
    "canonical_smiles",
    "image_uri",
    "cluster_label",
    "target_display",
    "molecular_weight",
    "logp",
    "qed",
]].astype(object).values

hovertemplate = "<br>".join(
    [
        "<b>%{customdata[0]}</b>",
        "SMILES: %{customdata[1]}",
        "Cluster: %{customdata[3]}",
        "Target: %{customdata[4]}",
        "MW: %{customdata[5]:.1f}",
        "LogP: %{customdata[6]:.2f}",
        "QED: %{customdata[7]:.2f}",
    ]
) + "<extra></extra>"

fig = go.Figure(
    go.Scattergl(
        x=gallery_df["component_1"],
        y=gallery_df["component_2"],
        mode="markers",
        marker=dict(
            size=8,
            color=color_values,
            colorscale="Viridis",
            colorbar=dict(title=color_title),
            opacity=0.9,
            line=dict(width=0),
        ),
        customdata=customdata,
        hovertemplate=hovertemplate,
    )
)
fig.update_layout(
    title="Interactive molecular landscape",
    xaxis_title="Component 1",
    yaxis_title="Component 2",
    template="plotly_dark",
    height=640,
    margin=dict(l=20, r=20, t=56, b=20),
)
plot_html = pio.to_html(fig, include_plotlyjs=True, full_html=False, div_id="molecule_map")

summary_html = f"""
<div style='margin:6px 0 10px 0;color:#94a3b8;'>
  Structure-only view built from Morgan fingerprints projected to two components. Hover or click any point to inspect the molecule before diving into the gallery and property sections.<br>
  Showing {len(gallery_df)} valid unique molecules after canonical deduplication.
</div>
"""

map_html = """
<div style='display:grid;grid-template-columns:minmax(0,1.35fr) 320px;gap:14px;align-items:start;'>
  <div style='background:#0f172a;border:1px solid #1e293b;border-radius:16px;padding:10px;'>
    __PLOT_HTML__
  </div>
  <div style='display:grid;gap:12px;'>
    <div style='background:#111827;border:1px solid #1f2937;border-radius:16px;padding:14px;color:#e5e7eb;'>
      <div style='font-size:18px;font-weight:700;'>Map guide</div>
      <div style='margin-top:8px;color:#94a3b8;font-size:13px;line-height:1.5;'>Each point is one canonically deduplicated molecule. Colors show the binary activity label, while cluster labels come from a lightweight K-means pass over the fingerprint space.</div>
    </div>
    <div style='background:#111827;border:1px solid #1f2937;border-radius:16px;padding:14px;color:#e5e7eb;'>
      <div style='font-size:16px;font-weight:700;margin-bottom:10px;'>Selected molecule</div>
      <div style='display:flex;justify-content:center;align-items:center;min-height:210px;border-radius:12px;background:#f8fafc;padding:8px;'>
        <img id='mol-img' alt='Molecule preview' style='max-width:100%;max-height:190px;'/>
      </div>
      <div style='margin-top:12px;font-size:13px;line-height:1.5;'>
        <div><strong>ID</strong>: <span id='panel-id'>-</span></div>
        <div><strong>Cluster</strong>: <span id='panel-cluster'>-</span></div>
        <div><strong>Target</strong>: <span id='panel-target'>-</span></div>
        <div><strong>MW</strong>: <span id='panel-mw'>-</span></div>
        <div><strong>LogP</strong>: <span id='panel-logp'>-</span></div>
        <div><strong>QED</strong>: <span id='panel-qed'>-</span></div>
        <div style='margin-top:10px;'><strong>SMILES</strong></div>
        <div id='panel-smiles' style='margin-top:4px;padding:8px;border-radius:10px;background:#0b1220;color:#cbd5e1;font-family:monospace;font-size:11px;word-break:break-all;'>-</div>
      </div>
    </div>
  </div>
</div>
<script>
const gd = document.getElementById('molecule_map');
function updatePanel(point) {
  if (!point || !point.customdata) return;
  document.getElementById('panel-id').textContent = point.customdata[0];
  document.getElementById('panel-smiles').textContent = point.customdata[1];
  document.getElementById('mol-img').src = point.customdata[2];
  document.getElementById('panel-cluster').textContent = point.customdata[3];
  document.getElementById('panel-target').textContent = point.customdata[4];
  document.getElementById('panel-mw').textContent = Number(point.customdata[5]).toFixed(1);
  document.getElementById('panel-logp').textContent = Number(point.customdata[6]).toFixed(2);
  document.getElementById('panel-qed').textContent = Number(point.customdata[7]).toFixed(2);
}
gd.on('plotly_hover', function(evt) { if (evt.points && evt.points.length) updatePanel(evt.points[0]); });
gd.on('plotly_click', function(evt) { if (evt.points && evt.points.length) updatePanel(evt.points[0]); });
if (gd.data && gd.data[0] && gd.data[0].customdata && gd.data[0].customdata.length) {
  updatePanel({customdata: gd.data[0].customdata[0]});
}
</script>
""".replace("__PLOT_HTML__", plot_html)

display(HTML(summary_html + map_html))


## 8. Molecular gallery


In [ ]:
import math
from io import BytesIO

def _mol_png_base64(mol, size=(180, 124)):
    image = Draw.MolToImage(mol, size=size)
    buffer = BytesIO()
    image.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode("ascii")

def render_gallery(df, page=1, page_size=8):
    page = max(1, int(page))
    page_size = max(1, int(page_size))
    total = len(df)
    total_pages = max(1, math.ceil(total / page_size))
    page = min(page, total_pages)
    start = (page - 1) * page_size
    end = start + page_size
    page_df = df.iloc[start:end]
    cards = []
    for idx, row in page_df.iterrows():
        img = _mol_png_base64(row["mol"])
        target_note = f"Target: {row[target_column]}" if target_column else ""
        smiles_note = row["canonical_smiles"][:28] + ("..." if len(row["canonical_smiles"]) > 28 else "")
        cards.append(
            f"<div style='background:#111827;border:1px solid #1f2937;border-radius:14px;padding:10px;'>"
            f"<img src='data:image/png;base64,{img}' style='width:100%;max-height:132px;object-fit:contain;border-radius:8px;background:white;'/>"
            f"<div style='padding-top:8px;font-family:monospace;color:#e5e7eb;font-size:11px;line-height:1.35;'>#{idx + 1}<br>{smiles_note}<br>{target_note}</div>"
            f"</div>"
        )
    html = f"""
    <div style='margin:8px 0 14px 0;'>
      <div style='font-size:16px;font-weight:600;'>Molecule gallery</div>
      <div style='margin-top:4px;color:#94a3b8;'>Page {page} of {total_pages} · {len(page_df)} molecules shown · {total} valid unique molecules after canonical deduplication.</div>
    </div>
    <div style='display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:10px;'>
      {''.join(cards)}
    </div>
    """
    display(HTML(html))

TARGET_FILTER = "all"  #@param ["all", "0", "1"]
SMILES_QUERY = ""  #@param {type:"string"}
PAGE = 1  #@param {type:"integer"}
PAGE_SIZE = 8  #@param [8, 12, 16, 24] {allow-input: true}

filtered_gallery_df = gallery_df.copy()
if TARGET_FILTER != "all" and target_column:
    filtered_gallery_df = filtered_gallery_df[filtered_gallery_df[target_column].astype(str) == TARGET_FILTER]
if SMILES_QUERY.strip():
    query = SMILES_QUERY.strip().lower()
    filtered_gallery_df = filtered_gallery_df[
        filtered_gallery_df[smiles_column].astype(str).str.lower().str.contains(query, na=False)
        | filtered_gallery_df["canonical_smiles"].astype(str).str.lower().str.contains(query, na=False)
    ]

render_gallery(filtered_gallery_df.reset_index(drop=True), page=PAGE, page_size=PAGE_SIZE)
print("Use the Colab controls at the top of this cell, then rerun it to change page, page size, or filters.")


## 9. Raw data analysis

In [ ]:
from IPython.display import display

print(f"Rows: {len(raw_df)}")
print(f"Columns: {len(raw_df.columns)}")
print(f"SMILES column: {smiles_column}")
print(f"Target column: {target_column}")
display(raw_df.head(8))

missing_summary = raw_df.isna().sum().sort_values(ascending=False)
missing_summary = missing_summary[missing_summary > 0]
if missing_summary.empty:
    print("No missing values detected in the raw tutorial dataset.")
else:
    display(missing_summary.head(10).rename("missing_count").to_frame())

In [ ]:
for name in ["dataset_overview.png", "missingness.png"]:
    path = eda_dir / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))

## 10. Target analysis

In [ ]:
if target_column is None:
    print("No target column found.")
else:
    display(raw_df[target_column].value_counts(dropna=False).rename("count").to_frame())

for name in ["target_distribution.png", "class_balance.png"]:
    path = eda_dir / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))

## 11. SMILES-derived properties analysis

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

property_df = gallery_df[[smiles_column, "canonical_smiles", "mol"] + ([target_column] if target_column else [])].copy()
property_df["molecular_weight"] = property_df["mol"].apply(Descriptors.MolWt)
property_df["logp"] = property_df["mol"].apply(Descriptors.MolLogP)
property_df["hbd"] = property_df["mol"].apply(Lipinski.NumHDonors)
property_df["hba"] = property_df["mol"].apply(Lipinski.NumHAcceptors)
property_df["qed"] = property_df["mol"].apply(QED.qed)
property_df["mw_pass"] = property_df["molecular_weight"] <= 500
property_df["logp_pass"] = property_df["logp"] <= 5
property_df["hbd_pass"] = property_df["hbd"] <= 5
property_df["hba_pass"] = property_df["hba"] <= 10
property_df["lipinski_pass"] = (
    property_df["mw_pass"]
    & property_df["logp_pass"]
    & property_df["hbd_pass"]
    & property_df["hba_pass"]
)
property_df["lipinski_pass_count"] = property_df[["mw_pass", "logp_pass", "hbd_pass", "hba_pass"]].sum(axis=1)
property_df["qed_band"] = pd.cut(
    property_df["qed"],
    bins=[-0.01, 0.30, 0.50, 0.67, 0.85, 1.01],
    labels=["<0.30", "0.30-0.50", "0.50-0.67", "0.67-0.85", ">0.85"],
)
scatter_df = property_df.sample(n=min(240, len(property_df)), random_state=42).copy()
display(property_df.head(6))

### Lipinski Rule of Five

In [ ]:
pass_all = int(property_df["lipinski_pass"].sum())
fail_any = int((~property_df["lipinski_pass"]).sum())
lipinski_html = f"""
<div style='margin:8px 0 14px 0;'>
  <div style='font-size:24px;font-weight:700;'>Lipinski Rule of Five</div>
  <div style='margin-top:8px;display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:12px;'>
    <div style='background:#111827;border:1px solid #1f2937;border-radius:14px;padding:14px;'>
      <div style='font-size:11px;color:#94a3b8;text-transform:uppercase;'>Pass all 4</div>
      <div style='font-size:26px;font-weight:700;color:#e5e7eb;'>{pass_all} ({pass_all / len(property_df) * 100:.1f}%)</div>
    </div>
    <div style='background:#111827;border:1px solid #1f2937;border-radius:14px;padding:14px;'>
      <div style='font-size:11px;color:#94a3b8;text-transform:uppercase;'>Fail at least 1</div>
      <div style='font-size:26px;font-weight:700;color:#e5e7eb;'>{fail_any} ({fail_any / len(property_df) * 100:.1f}%)</div>
    </div>
  </div>
  <div style='margin-top:10px;color:#94a3b8;'>Classical Lipinski compliance is shown as a strict 4-of-4 score using molecular weight, LogP, H-bond acceptors, and H-bond donors.</div>
</div>
"""
display(HTML(lipinski_html))

In [ ]:
rule_names = ["Molecular weight ≤ 500", "LogP ≤ 5", "H-bond acceptors ≤ 10", "H-bond donors ≤ 5"]
rule_pass_counts = [
    int(property_df["mw_pass"].sum()),
    int(property_df["logp_pass"].sum()),
    int(property_df["hba_pass"].sum()),
    int(property_df["hbd_pass"].sum()),
]
rule_fail_counts = [len(property_df) - value for value in rule_pass_counts]
pass_count_distribution = property_df["lipinski_pass_count"].value_counts().reindex([4, 3, 2, 1, 0], fill_value=0)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes[0, 0].bar(["Pass all 4", "Fail at least 1"], [pass_all, fail_any], color=["#6ee7b7", "#f59e0b"])
axes[0, 0].set_title("Pass all four rules")
axes[0, 0].set_xlabel("Compliance bucket")
axes[0, 0].set_ylabel("Molecules")

axes[0, 1].bar([f"{idx} of 4" for idx in pass_count_distribution.index], pass_count_distribution.values, color="#60a5fa")
axes[0, 1].set_title("How many of the four rules each molecule passes")
axes[0, 1].set_xlabel("Pass count")
axes[0, 1].set_ylabel("Molecules")

axes[1, 0].barh(rule_names, rule_pass_counts, color="#c084fc")
axes[1, 0].set_title("Per rule compliance")
axes[1, 0].set_xlabel("Molecules")

axes[1, 1].barh(rule_names, rule_fail_counts, color="#fb7185")
axes[1, 1].set_title("Per rule failure")
axes[1, 1].set_xlabel("Molecules")

fig.tight_layout()
plt.show()

### Drug-Likeness (QED)

In [ ]:
mean_qed = property_df["qed"].mean()
median_qed = property_df["qed"].median()
qed_ge_067 = int((property_df["qed"] >= 0.67).sum())
qed_ge_085 = int((property_df["qed"] >= 0.85).sum())
qed_html = f"""
<div style='margin:8px 0 14px 0;'>
  <div style='font-size:24px;font-weight:700;'>Drug-Likeness (QED)</div>
  <div style='margin-top:8px;display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:12px;'>
    <div style='background:#111827;border:1px solid #1f2937;border-radius:14px;padding:14px;'><div style='font-size:11px;color:#94a3b8;text-transform:uppercase;'>Mean QED</div><div style='font-size:26px;font-weight:700;color:#e5e7eb;'>{mean_qed:.2f}</div></div>
    <div style='background:#111827;border:1px solid #1f2937;border-radius:14px;padding:14px;'><div style='font-size:11px;color:#94a3b8;text-transform:uppercase;'>Median QED</div><div style='font-size:26px;font-weight:700;color:#e5e7eb;'>{median_qed:.2f}</div></div>
    <div style='background:#111827;border:1px solid #1f2937;border-radius:14px;padding:14px;'><div style='font-size:11px;color:#94a3b8;text-transform:uppercase;'>QED ≥ 0.67</div><div style='font-size:26px;font-weight:700;color:#e5e7eb;'>{qed_ge_067}</div></div>
    <div style='background:#111827;border:1px solid #1f2937;border-radius:14px;padding:14px;'><div style='font-size:11px;color:#94a3b8;text-transform:uppercase;'>QED ≥ 0.85</div><div style='font-size:26px;font-weight:700;color:#e5e7eb;'>{qed_ge_085}</div></div>
  </div>
  <div style='margin-top:10px;color:#94a3b8;'>Scatter plots use a fixed-seed sample of {len(scatter_df)} molecules out of {len(property_df)} profiled molecules.</div>
</div>
"""
display(HTML(qed_html))

In [ ]:
qed_band_counts = property_df["qed_band"].value_counts().reindex(["<0.30", "0.30-0.50", "0.50-0.67", "0.67-0.85", ">0.85"], fill_value=0)
band_palette = {
    "<0.30": "#ef4444",
    "0.30-0.50": "#f59e0b",
    "0.50-0.67": "#facc15",
    "0.67-0.85": "#34d399",
    ">0.85": "#22c55e",
}

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
sns.histplot(property_df["qed"], bins=24, ax=axes[0, 0], color="#fbbf24")
axes[0, 0].set_title("QED distribution")
axes[0, 0].set_xlabel("QED")
axes[0, 0].set_ylabel("Molecules")

axes[0, 1].bar(qed_band_counts.index.astype(str), qed_band_counts.values, color=[band_palette[str(label)] for label in qed_band_counts.index])
axes[0, 1].set_title("Drug-likeness bands")
axes[0, 1].set_xlabel("QED band")
axes[0, 1].set_ylabel("Molecules")

scatter_colors = scatter_df["qed_band"].astype(str).map(band_palette).fillna("#60a5fa")
axes[1, 0].scatter(scatter_df["molecular_weight"], scatter_df["qed"], c=scatter_colors, alpha=0.8, s=18)
axes[1, 0].set_title("QED vs molecular weight")
axes[1, 0].set_xlabel("Molecular weight")
axes[1, 0].set_ylabel("QED")

axes[1, 1].scatter(scatter_df["logp"], scatter_df["qed"], color="#60a5fa", alpha=0.8, s=18)
axes[1, 1].set_title("QED vs LogP")
axes[1, 1].set_xlabel("LogP")
axes[1, 1].set_ylabel("QED")

fig.tight_layout()
plt.show()